> **교육생용 정답 코드**
> Python 3 커널을 선택하고 위에서부터 셀을 실행하세요.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False
plt.rc("font", family="Malgun Gothic")


# [실습] 서울시 미세먼지 데이터로 공익 애플리케이션 만들기 - 데이터 전처리

서울시의 미세먼지 데이터를 활용하여, 공익 애플리케이션을 만들어봅니다.

## 1. 데이터 불러오기

실습환경에 저장된 데이터를 불러옵니다.   
데이터 출처: [공공데이터 포털 서울시 미세먼지 데이터](https://www.data.go.kr/data/15089266/fileData.do)

In [ ]:
import os
import pandas as pd

데이터를 다운받아 압축을 해제하면, 아래와 같은 폴더로 정리되어 있습니다. (플랫폼에는 별도 준비 필요x)

In [ ]:
data_dir = "Seoul_20221231"
filelist = os.listdir(data_dir)
filelist

데이터를 불러와서 하나로 합쳐줍니다. 공공데이터는 다양한 encoding으로 저장되어있을 수 있으니, 파일을 불러올 때 주의합니다.

In [ ]:
df = pd.DataFrame()
for i in filelist:
    df = pd.concat([df,pd.read_csv(os.path.join(data_dir,i),encoding="euckr")],axis=0)
df

## 2. 데이터 전처리하기 & Streamlit 용 데이터 저장
기본적으로 날짜 데이터는 문자열로 로딩됩니다. 데이터를 사용하기 좋도록 시간 값을 날짜형식으로 바뀌줍니다.

In [ ]:
from datetime import datetime
import re

df["일시"] = df["일시"].apply(lambda x : re.sub(" ([0-9]):"," 0\\1:",x)).apply(datetime.fromisoformat)
df

데이터를 정렬해줍니다.

In [ ]:
df = df.sort_values(by=["일시","구분"])
df

In [ ]:
df.reset_index(inplace=True)
# df.to_csv("서울시 미세먼지.csv",index=False)
del df["index"]
df

데이터의 컬럼 이름이 중간에 변했습니다. 초미세먼지(PM25) ->  초미세먼지(PM2.5)    
해당 형식을 맞춰줍니다.

In [ ]:
df["초미세먼지(PM2.5)"] = df.where(pd.notnull(df),df['초미세먼지(PM25)'],axis=0)["초미세먼지(PM2.5)"]
del df['초미세먼지(PM25)']
df

초 미세먼지에 대한 전체 데이터를 그려봅니다.

In [ ]:
import plotly.express  as px

# 시간이 조금 걸립니다.
fig = px.line(df,x="일시",y="초미세먼지(PM2.5)",
              title="서울시 구별 미세먼지",
              color='구분',
              line_group='구분',
              hover_name="구분")
fig

시기상 오래 전이고, 이상치도 많아보이는 2014년 이전 데이터는 분석에서 제외합니다.

In [ ]:
df = df[df["일시"]> datetime(2014,1,1)]

fig = px.line(df,x="일시",y="초미세먼지(PM2.5)",
              title="서울시 구별 미세먼지",
              color='구분',
              line_group='구분',
              hover_name="구분")
fig

지역별 정도 확인을 위한 그래프를 그리기 위하여 pivot table을 만듭니다.

In [ ]:
pivot_table = pd.pivot_table(df,index="구분",values=["미세먼지(PM10)","초미세먼지(PM2.5)"],aggfunc="mean")

In [ ]:
pivot_table["지역구"] = pivot_table.index

In [ ]:
fig = px.bar(pivot_table, y='미세먼지(PM10)',color="지역구")
fig.show()

해당 데이터를 Streamlit에서 사용하기 위해 저장합니다.

In [ ]:
from pathlib import Path

output = Path("../seoul.csv.gz")
temporary_output = output.with_name("seoul.tmp.csv.gz")
df.to_csv(temporary_output, encoding="utf-8-sig", index=False)
temporary_output.replace(output)